# Solution Analysis

This notebook summarizes the historical score trajectory and inspects which engineered features look most relevant for the classification and regression tasks.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_selection import f_classif, f_regression

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from creditsense_public.artifacts import load_report_table
from creditsense_public.features import add_features, load_competition_data

stage_df = load_report_table("stage_progress.csv")
leaderboard_df = load_report_table("model_leaderboard.csv")
feature_stage_df = load_report_table("feature_engineering_stages.csv")
stage_df


,stage,combined,category,summary
0,Assignment baseline,0.510000,baseline,Starter notebook with a simple linear baseline...
1,Polynomial parallel blend,0.822506,feature_engineering,"Shared finance ratios, semantic missingness fl..."
2,Polynomial plus MPS blend,0.834266,feature_engineering,Added the first small MPS branch on top of the...
3,Boosted overnight meta ensemble,0.849489,public_best,"XGBoost, LightGBM, CatBoost, ExtraTrees, and a..."
4,Target-stats bagging,0.855731,local_research,Out-of-fold target statistics and bagging push...
5,MLP meta stack,0.857181,local_research,A compact neural lane improved the final combi...
6,Final weighted local blend,0.857645,local_best,The strongest local blend used a small weighte...


In [2]:
leaderboard_df.sort_values("combined", ascending=False)

,model,accuracy,r2,combined,family
4,mlp_weighted_blend_v1,0.860000,0.855291,0.857645,local_best
3,mlp_meta_stack_v1,0.858714,0.855648,0.857181,local_research
2,target_stats_blend,0.856286,0.855177,0.855731,local_research
0,meta_overnight,0.847714,0.851263,0.849489,public_best
1,blend_overnight,0.842429,0.848508,0.845468,public_best


In [3]:
feature_stage_df

,addition,validation_combined,why_it_helped
0,Domain ratios and semantic missingness flags,0.822506,"Affordability, utilization, and context-aware ..."
1,Controlled polynomial interactions,0.825140,A small interaction set captured stress combin...
2,Tree ensemble diversification,0.849489,Different tree families made different mistake...
3,Target statistics and bagging,0.855731,Target-aware categorical summaries helped the ...
4,Neural branch and weighted fusion,0.857645,The neural branch added smooth non-linear stru...


In [4]:
data = load_competition_data()
engineered = add_features(data["train_df"].drop(columns=["RiskTier", "InterestRate"]))
numeric_engineered = engineered.select_dtypes(include=[np.number]).fillna(0.0)
cls_scores, _ = f_classif(numeric_engineered, data["train_df"]["RiskTier"].to_numpy())
reg_scores, _ = f_regression(numeric_engineered, data["train_df"]["InterestRate"].to_numpy())
signal_df = pd.DataFrame({
    "feature": numeric_engineered.columns,
    "classification_signal": np.nan_to_num(cls_scores, nan=0.0, posinf=0.0, neginf=0.0),
    "regression_signal": np.nan_to_num(reg_scores, nan=0.0, posinf=0.0, neginf=0.0),
})
signal_df.nlargest(12, "classification_signal")[["feature", "classification_signal"]]

,feature,classification_signal
25,NumberOfLatePayments30Days,6677.758726
62,LatePaymentTotal,6631.931183
63,LatePaymentSeverity,5234.047142
24,RevolvingUtilizationRate,4966.924555
97,LateSeverityXUtilization,4761.809897
28,NumberOfChargeOffs,4474.034887
98,LateSeverityXDebtToIncome,4041.726086
99,DerogatoryXUtilization,3950.162383
90,DerogatorySeverity,3893.251024
91,LatePaymentPerOpenAccount,3841.871325


In [5]:
signal_df.nlargest(12, "regression_signal")[["feature", "regression_signal"]]

,feature,regression_signal
97,LateSeverityXUtilization,30506.626799
28,NumberOfChargeOffs,25386.312224
62,LatePaymentTotal,23095.646800
63,LatePaymentSeverity,22307.481216
98,LateSeverityXDebtToIncome,20641.250075
25,NumberOfLatePayments30Days,18465.647044
99,DerogatoryXUtilization,18236.450655
91,LatePaymentPerOpenAccount,17316.014059
27,NumberOfLatePayments90Days,15179.358289
90,DerogatorySeverity,10317.454537


# Takeaway

The strongest features overlap, but not perfectly. Delinquency severity and utilization interactions dominate the classification side, while affordability and asset coverage features matter more for the interest-rate regression.